# Tensores, datasets y lotes con PyTorch

Retomaremos las matrices de Titanic y después examinaremos imágenes de CIFAR-10.
El objetivo es reconocer qué contiene una muestra, cómo se transforma y cómo se
agrupa con otras. No entrenaremos un modelo.

## Contenido

1. Convertir arreglos NumPy a tensores.
2. Relacionar muestras y etiquetas con `Dataset` y `DataLoader`.
3. Cargar CIFAR-10 y transformar imágenes.
4. Interpretar las dimensiones de un lote y normalizar sus valores.
5. Guardar una selección para la sesión de visualización.

Abre esta notebook desde `sesion-03-pytorch`. La [preparación del proyecto](./README.md)
genera las salidas de Titanic y descarga CIFAR-10 antes de la clase.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2

project_dir = Path.cwd()
titanic_dir = project_dir.parent / "sesion-02-pandas" / "outputs"
if not (project_dir / "data_lab").is_dir():
    raise FileNotFoundError("Start this notebook from sesion-03-pytorch")
if not (titanic_dir / "features.npy").is_file():
    raise FileNotFoundError(
        "Run the session 2 main.py first to generate the Titanic arrays"
    )

## 1. Del arreglo al tensor

Un tensor de PyTorch organiza valores por dimensiones, igual que los arreglos
que ya conocemos. PyTorch añade herramientas para trabajar con aceleradores y
calcular derivadas durante el entrenamiento. Aquí nos interesa su representación
de datos: `shape`, `dtype` y `device`, el dispositivo donde residen los valores.

`torch.from_numpy` conserva el tipo del arreglo. Las variables de Titanic serán
`float32` y sus etiquetas `int64`. `dim` indica el eje de una operación, como
`axis` en NumPy. Los siguientes tensores se crean en CPU. `.numpy()` permite consultarlos
como arreglos y `.item()` extrae un número de un tensor de un solo elemento.

[Introducción oficial a los tensores](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).

In [ ]:
features_np = np.load(titanic_dir / "features.npy", allow_pickle=False)
labels_np = np.load(titanic_dir / "labels.npy", allow_pickle=False)
features = torch.from_numpy(features_np)
labels = torch.from_numpy(labels_np)
print("Features:", features.shape, features.dtype, features.device)
print("Labels:", labels.shape, labels.dtype)
print("Mean by feature:", features.mean(dim=0))
np.testing.assert_allclose(
    features.mean(dim=0).numpy(), features_np.mean(axis=0), rtol=1e-5
)
assert features.shape == (891, 4)
assert labels.shape == (891,)

### Memoria compartida

`from_numpy` comparte memoria con el arreglo de origen. Si modificas uno,
cambia el otro. `.clone()` crea una copia del tensor. El experimento usa datos
pequeños para conservar intactas las variables de Titanic.

[`torch.from_numpy`](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html).

In [ ]:
values_np = np.array([10.0, 20.0], dtype=np.float32)
shared = torch.from_numpy(values_np)
independent = shared.clone()
shared[0] = 99
print("NumPy:", values_np)
print("Independent tensor:", independent)
assert independent[0].item() == 10.0

### Ejercicio 1: una selección independiente

Crea una copia de las primeras tres filas de `features`, cambia su primera
posición y comprueba que `features` y `features_np` conservan sus valores.
Verifica la forma `(3, 4)`. Usa `.clone()` y las comprobaciones de NumPy que ya
conoces; puedes consultar un tensor de CPU con `.numpy()`.

In [ ]:
# Write your solution here.

## 2. Una muestra y un lote

Un `Dataset` permite obtener muestras. En los datasets indexables de esta sesión,
`len(dataset)` indica su cantidad y `dataset[i]` obtiene una muestra.
`TensorDataset` empareja la fila `i` de cada tensor: variables y etiqueta deben
pertenecer al mismo pasajero y tener la misma cantidad de filas.

[Datasets y DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

In [ ]:
passenger_dataset = TensorDataset(features, labels)
passenger_features, passenger_label = passenger_dataset[0]
print("Samples:", len(passenger_dataset))
print("One sample:", passenger_features.shape, passenger_label.item())

`DataLoader` agrupa muestras en lotes. `batch_size` fija cuántas incluir y
`shuffle=False` conserva el orden. `num_workers=0` carga en el mismo proceso.
`drop_last=False` conserva el último lote aunque sea más pequeño.

`Subset` permite trabajar con algunos índices. En este ejemplo, diez muestras
con lotes de cuatro producen tamaños 4, 4 y 2.

In [ ]:
small_dataset = Subset(passenger_dataset, range(10))
loader = DataLoader(
    small_dataset, batch_size=4, shuffle=False, drop_last=False, num_workers=0
)
for batch_features, batch_labels in loader:
    print(batch_features.shape, batch_labels.shape)

### Ejercicio 2: el último lote

Recorre `small_dataset` con `batch_size=3`. Anota los tamaños de los lotes y
cuántas muestras se procesan en total. Repite con `drop_last=True` y explica
qué muestra deja de entrar. Conserva `shuffle=False` en ambos recorridos.

In [ ]:
# Write your solution here.

### Mezclar el orden de forma repetible

`shuffle=True` cambia el orden, manteniendo juntos los componentes de cada
muestra. Un `torch.Generator` con semilla permite repetir ese orden en el mismo
entorno. Creamos un generador nuevo para cada recorrido: reutilizar uno avanza
su estado y puede dar otro orden.

`torch.arange(10)` crea los índices de 0 a 9. `next(iter(loader))` obtiene
un lote; el `[0]` extrae su único componente porque este dataset tiene un solo
tensor. Mostramos índices pequeños para que el efecto se vea. La semilla no garantiza
resultados idénticos entre todas las versiones y plataformas.

[Reproducibilidad](https://docs.pytorch.org/docs/stable/notes/randomness.html).

In [ ]:
sample_ids = TensorDataset(torch.arange(10))
orders = []
for _ in range(2):
    generator = torch.Generator().manual_seed(42)
    shuffled_loader = DataLoader(
        sample_ids, batch_size=10, shuffle=True, generator=generator, num_workers=0
    )
    ids = next(iter(shuffled_loader))[0]
    orders.append(ids.tolist())
print(orders)
assert orders[0] == orders[1]

## 3. CIFAR-10: imágenes y clases

CIFAR-10 contiene imágenes RGB de 32 × 32 píxeles en diez clases. Tiene 50 000
imágenes de entrenamiento y 10 000 de prueba. Usaremos el conjunto de prueba
(`train=False`) para inspección, sin evaluar ni entrenar modelos.

`root` indica la carpeta local y `download=False` reutiliza la descarga preparada.
Sin transformación, cada muestra devuelve una imagen de Pillow y el identificador
entero de su clase. `classes` relaciona esos identificadores con sus nombres.
Ampliamos la imagen solo para verla en pantalla.

[Fuente de CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) ·
[API de CIFAR10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html).

In [ ]:
raw_dataset = CIFAR10(root=project_dir / "data", train=False, download=False)
raw_image, class_id = raw_dataset[0]
print("Samples:", len(raw_dataset))
print("Image size:", raw_image.size, "Mode:", raw_image.mode)
print("Class:", class_id, raw_dataset.classes[class_id])
display(raw_image.resize((192, 192)))

### Transformar una imagen

La imagen como arreglo tiene orden `(alto, ancho, canales)`. PyTorch suele
trabajar con `(canales, alto, ancho)`. `v2.ToImage()` cambia la representación a
un tensor de imagen y `v2.ToDtype(torch.float32, scale=True)` convierte los
enteros entre 0 y 255 a flotantes entre 0 y 1. `v2.Compose` aplica los pasos en
orden. Cambiar solo el tipo no equivale a cambiar la escala.

[Transformaciones de torchvision](https://docs.pytorch.org/vision/stable/transforms.html).

In [ ]:
image_array = np.array(raw_image)
image_uint8 = v2.ToImage()(raw_image)
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
image = transform(raw_image)
print("Array:", image_array.shape, image_array.dtype)
print("Image tensor:", image_uint8.shape, image_uint8.dtype)
print("Scaled tensor:", image.shape, image.dtype)
print("Value range:", image.min().item(), image.max().item())
assert image.shape == (3, 32, 32)

## 4. Dimensiones de un lote de imágenes

Pasamos la transformación a `CIFAR10`: se aplica cuando pedimos cada muestra.
El `DataLoader` añade un eje inicial para las imágenes del lote:
`(N, C, H, W)` corresponde a cantidad de imágenes, canales, alto y ancho.
Las etiquetas quedan en un vector `(N,)`.

`next(iter(loader))` obtiene el primer lote. Reducir `dim=(2, 3)` promedia alto
y ancho y conserva una media por canal para cada imagen.

In [ ]:
cifar = CIFAR10(
    root=project_dir / "data", train=False, download=False, transform=transform
)
image_loader = DataLoader(cifar, batch_size=16, shuffle=False, num_workers=0)
images, image_labels = next(iter(image_loader))
print("Images:", images.shape)
print("Labels:", image_labels.shape, image_labels.dtype)
print("First class:", cifar.classes[image_labels[0].item()])
image_channel_means = images.mean(dim=(2, 3))
print("Mean per image and channel:", image_channel_means.shape)
assert images.shape == (16, 3, 32, 32)

### Ejercicio 3: una media por canal

Calcula una sola media por canal para las 16 imágenes del lote. El resultado
debe tener forma `(3,)`. Puedes promediar `image_channel_means` sobre las
imágenes o reducir directamente los ejes necesarios de `images`. Compara ambos
cálculos con `torch.testing.assert_close`, que comprueba cercanía numérica como
`assert_allclose` en NumPy. Explica qué ejes desaparecen.

In [ ]:
# Write your solution here.

### Normalizar valores

`Normalize` aplica `(valor - mean) / std` a cada canal. Con `mean=0.5` y
`std=0.5` para los tres canales, el intervalo `[0, 1]` pasa a `[-1, 1]`.
Son constantes elegidas para este ejemplo: no son estadísticas calculadas de
CIFAR-10 y no garantizan que el lote tenga media cero y desviación uno.

La operación devuelve un tensor nuevo con la misma forma. Guardaremos para la
siguiente sesión las imágenes en `[0, 1]`, que son más directas de visualizar.

[`Normalize`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.Normalize.html).

In [ ]:
normalize = v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
normalized_images = normalize(images)
print("Original range:", images.min().item(), images.max().item())
print(
    "Normalized range:", normalized_images.min().item(), normalized_images.max().item()
)
assert normalized_images.shape == images.shape

### Ejercicio 4: recuperar la escala

Invierte la transformación anterior para recuperar valores en `[0, 1]`.
Guarda el resultado en `restored_images` y compáralo con `images` mediante
`torch.testing.assert_close`. Explica por qué conservar la forma del tensor
no significa conservar sus valores.

In [ ]:
# Write your solution here.

## 5. Reutilizar y guardar los datos

La función `collect_samples` del módulo usa `Subset` y `DataLoader` para recorrer
las primeras muestras. Une los lotes con `torch.cat(..., dim=0)`: concatena sobre
un eje existente. En el ejemplo siguiente, dos grupos de dos imágenes recuperan
un tensor de cuatro imágenes.

In [ ]:
first_part = images[:2]
second_part = images[2:4]
joined = torch.cat([first_part, second_part], dim=0)
print(joined.shape)
torch.testing.assert_close(joined, images[:4])

Para contar clases, `torch.bincount` cuenta cuántas veces aparece cada
identificador. `minlength=10` incluye las diez clases aunque alguna no aparezca
en la selección. El módulo añade sus nombres para escribir el CSV.

Estos conteos describen las primeras 64 imágenes, no el conjunto completo.

In [ ]:
from data_lab.inspection import collect_samples, image_transform, label_summary

torch.testing.assert_close(image_transform()(raw_image), image)
selected_images, selected_labels = collect_samples(cifar, limit=64, batch_size=16)
counts = torch.bincount(selected_labels, minlength=len(cifar.classes))
print("Selected images:", selected_images.shape)
print("Counts:", counts.tolist())
print(label_summary(selected_labels, cifar.classes))
assert counts.sum().item() == 64

Desde la terminal, ejecuta:

```bash
uv run --locked python main.py
```

La aplicación guarda `images.npy`, `labels.npy`, `class_counts.csv` y
`report.json` en `outputs/`. El reporte describe la selección, el orden de ejes
y los nombres de clases. `--limit` y `--batch-size` permiten cambiar la cantidad
de muestras y el tamaño de lote. Por ejemplo, `--limit 10 --batch-size 4`
conserva también las dos muestras del último lote.

Continúa con el [ejercicio 5](./PRACTICA.md), que amplía el resumen por clase.
Antes de entregar, reinicia el kernel y ejecuta todas las celdas.